In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Time Series
## ARIMA 

Уравнение модели авторегрессии (AR)
$$ x_t = const + \sum_{i=1}^p\beta_ix_{t-i} + \epsilon_t$$

Гиперпараметры:
| p | d | q |
| :-: | :-: | :-: |
| Количество предыдущих значений, участвующих в AR модели | Какое количество раз надо взять разность между соседними точками, чтобы "убрать тренд", тем самым сделав ряд стационарным | Количество прошлых ошибок, которые надо помнить |

### 4.8        (Time Series)  A classic dataset used in time series lectures is the quarterly earnings per share (EPS) for Johnson & Johnson stock.  Shumway and Stoffer [2006] utilize this dataset in their text and provide a copy of the dataset.   

### There are 84 values in this dataset covering the EPS from the first quarter of 1960 to the fourth quarter of 1980.

In [ ]:
# Смотрю, как вообще устроены данные
!head resources/8-lab/jj.dat
!tail resources/8-lab/jj.dat

In [ ]:
df = pd.read_csv('resources/8-lab/jj.dat', header=None).astype(np.float32)
df.columns = ['EPS']
df['time'] = pd.DataFrame([i for i in range(84)]).astype(np.int8)
df


##  a.      After obtaining a copy of the dataset, plot the quarterly EPS vs. time.  Describe any patterns that you observe.

Паттерны:
- Долгосрочный восходящий тренд - с каждым годом акция всё больше растёт
- Яркая сезонность - В рамках одного года (4 точки соседние) очень хороший разброс, особенно в последние годы
- Растущая амплитуда колебаний - с каждым годом ```max(EPS)-min(EPS)``` всё больше

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df['time'], df['EPS'], marker='o', color='orange', markersize=4)
plt.xlabel('Time')
plt.ylabel('EPS')
plt.grid()

plt.show()

## b.      In order to perform an ARIMA model, the time series will need to be transformed to remove any trend.  Plot the difference of xt and xt-1, for all t > 0.   Has this difference adequately detrended the series? Does the variability of the EPS appear constant over time?

- Взятие разности помогло устранить восходящий тренд
- Дисперсия не является постоянной: она растёт со временем

### Why does the constant variance matter?

Метод максимального правдоподобия требует константной дисперсии
$$f(\varepsilon_t) = \frac{1}{\sqrt{2\pi \sigma^2}} \exp\left( -\frac{\varepsilon_t^2}{2\sigma^2} \right)$$
$$L = f(\varepsilon_1) \cdot f(\varepsilon_2) \cdot \dots \cdot f(\varepsilon_T)$$
Если дисперсия не будет постоянной, то для каждого $$f(\epsilon_i)$$ будет своя $$\sigma_i$$, что повлечет дополнительные степени свободы, на разрешение которых не хватит точек во временном ряду (Следовтельно система неразрешима)
Потому алгоритм и написан для "постоянной дисперсии", на уровне отработки ```ARIMA.fit()``` просто результат будет неправильным, тк мы заменим p сигм на какую-то одну 

In [ ]:
diffs = df['EPS'].diff()

plt.figure(figsize=(12, 6))

plt.plot(df['time'], diffs, marker='o', color='green', markersize=4)
plt.xlabel('Time')
plt.ylabel('EPS')
plt.grid()

plt.show()

## c.       Plot the log10 of the quarterly EPS vs. time and plot the difference of log10(xt ) and log10(xt-1) for all t > 0

In [ ]:
log_ = np.log10(df['EPS'])
log_diffs = log_.diff()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].plot(df['time'], log_, marker='o', color='orange', markersize=4)
axes[0].set_title('Log10(EPS) vs Time')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Log10(EPS)')
axes[0].grid()

axes[1].plot(df['time'], log_diffs, marker='o', color='green', markersize=4)
axes[1].set_title('Log10(EPS).diff() vs Time ')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Log10(EPS).diff()')
axes[1].grid()

plt.show()

###  Has this adequately detrended the series?
Да, взятие разности первого порядка устранило тренд

### Has the variability of the differenced log10(EPS) become more constant? 
Дисперсия стала более постоянной, причем можно заметить, что можно разделить её на 3 части\
от 0 до 20; от 20 до 60; от 60 до 84

## d.      Treating the differenced log10 of the EPS series as a stationary series, plot the ACF and PACF of this series.  What possible ARIMA models would you consider and why?

- ```ACF``` - Автокорреляционная функция. Измеряет корреляцию между текущим моментом времени i и прошлым моментом времени i - k, где k - лаг (Какой шаг назад во времени делаем)
- ```PACF``` - Частная автокорреляционная функция ** ДОДЕЛАТЬ **

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# ACF
plot_acf(log_diffs.dropna(), ax=axes[0], lags=40)
axes[0].set_title('Autocorrelation Function')

# PACF
plot_pacf(log_diffs.dropna(), ax=axes[1], lags=40)
axes[1].set_title('Partial Autocorrelation Function')

plt.tight_layout()
plt.show()